In [1]:
#Imports and configurations

import pandas as pd
import datetime
import holidays
import numpy as np

estonian_holidays = holidays.Estonia(years=[2025, 2026])

In [2]:
#Reading the data

df_spot_price = pd.read_csv("../Data/spot_market_price.csv")
df_consumption = pd.read_csv("../Data/consumption.csv")
df_fixed_plan = pd.read_csv("../Data/fixed_plans.csv")
df_spot_plan = pd.read_csv("../Data/spot_plans.csv")

In [4]:
#Converting the "timestamp" column values from str to datetime data type

df_spot_price["timestamp"] = pd.to_datetime(df_spot_price["timestamp"], dayfirst = True)
df_consumption["timestamp"] = pd.to_datetime(df_consumption["timestamp"], dayfirst = True)

In [5]:
#Adding an hour column to the dataframes

df_consumption["hour"] = df_consumption["timestamp"].dt.hour
df_spot_price["hour"] = df_spot_price["timestamp"].dt.hour

In [6]:
#Adding a new column to check if it's daytime (between 7:00 and 22:00)

df_consumption["is_daytime"] = (df_consumption["hour"] >= 7) & (df_consumption["hour"] < 22)
df_spot_price["is_daytime"] = (df_spot_price["hour"] >= 7) & (df_spot_price["hour"] < 22)

In [7]:
#Adding a day column to the dataframes

df_consumption["day_of_week"] = df_consumption["timestamp"].dt.dayofweek
df_spot_price["day_of_week"] = df_spot_price["timestamp"].dt.dayofweek

In [8]:
#Adding a new column to check if it's weekend (Saturday or Sunday)

df_consumption["is_weekend"] = (df_consumption["day_of_week"] >= 5)
df_spot_price["is_weekend"] = (df_spot_price["day_of_week"] >= 5)

In [9]:
#Adding a new column to check if it's Estonian national holiday

df_consumption['is_public_holiday'] = df_consumption['timestamp'].dt.date.isin(estonian_holidays)
df_spot_price['is_public_holiday'] = df_spot_price['timestamp'].dt.date.isin(estonian_holidays)

In [10]:
#Adding a new column to check if it's required to use night rate (between 22:00 and 7:00, whole weekends and national holidays)

df_consumption['is_night_rate'] = ~df_consumption['is_daytime'] | df_consumption['is_weekend'] | df_consumption['is_public_holiday']
df_spot_price['is_night_rate'] = ~df_spot_price['is_daytime'] | df_spot_price['is_weekend'] | df_spot_price['is_public_holiday']

In [11]:
df_consumption.dtypes

household_id                    str
timestamp            datetime64[us]
date                            str
time                            str
consumption_kwh             float64
hour                          int32
is_daytime                     bool
day_of_week                   int32
is_weekend                     bool
is_public_holiday              bool
is_night_rate                  bool
dtype: object

In [12]:
df_consumption_and_spot_price = pd.merge(df_consumption, df_spot_price, on = "timestamp", how = "left")

In [13]:
df_consumption_and_spot_price = df_consumption_and_spot_price.drop(columns = ["date_y", "time_y", "hour_y", "is_daytime_y", "day_of_week_y", "is_weekend_y", "is_public_holiday_y", "is_night_rate_y"])

In [14]:
df_consumption_and_spot_price = df_consumption_and_spot_price.rename(columns = {"date_x": "date", "time_x": "time", "hour_x": "hour", "is_daytime_x": "is_daytime", "day_of_week_x": "day_of_week", "is_weekend_x": "is_weekend", "is_public_holiday_x": "is_public_holiday", "is_night_rate_x": "is_night_rate"})

In [15]:
df = df_consumption_and_spot_price

In [17]:
results = []

# Alternative for loop for the fixed plans

for _, plan in df_fixed_plan.iterrows():
    cost_per_interval = plan['monthly_price'] / (30.4375 * 24 * 4)
    
    daytime_cost = np.where(
        ~df['is_night_rate'],
        plan['kwh_price_day'] * df['consumption_kwh'],
        0
    )
    
    nighttime_cost = np.where(
        df['is_night_rate'],
        plan['kwh_price_night'] * df['consumption_kwh'],
        0
    )

    df_plan = df[['timestamp', 'household_id', 'consumption_kwh']].copy()
    df_plan['plan_id'] = plan['plan_id']
    df_plan['plan_name'] = plan['electricity_plan_name']
    df_plan['vendor'] = plan['electricity_vendor_name']
    df_plan['plan_type'] = 'fixed'
    df_plan['production_type'] = plan['production_type']
    df_plan['contract_length_months'] = plan['contract_length_months']
    df_plan['is_ending_fee'] = plan['is_ending_fee']
    df_plan['daytime_cost'] = daytime_cost
    df_plan['nighttime_cost'] = nighttime_cost
    df_plan['margin_cost'] = 0
    df_plan['spot_energy_cost'] = 0
    df_plan['monthly_cost'] = cost_per_interval
    df_plan['total_cost'] = daytime_cost + nighttime_cost + cost_per_interval

    results.append(df_plan)

# Alternative for loop for the spot plans

for _, plan in df_spot_plan.iterrows():
    cost_per_interval = plan['monthly_price'] / (30.4375 * 24 * 4)
    
    spot_energy_cost = df['spot_price_incl_vat'] * df['consumption_kwh']
    margin_cost = plan['margin_price'] * df['consumption_kwh']

    df_plan = df[['timestamp', 'household_id', 'consumption_kwh']].copy()
    df_plan['plan_id'] = plan['plan_id']
    df_plan['plan_name'] = plan['electricity_plan_name']
    df_plan['vendor'] = plan['electricity_vendor_name']
    df_plan['plan_type'] = 'spot'
    df_plan['production_type'] = plan['production_type']
    df_plan['contract_length_months'] = None
    df_plan['is_ending_fee'] = plan['is_ending_fee']
    df_plan['daytime_cost'] = 0
    df_plan['nighttime_cost'] = 0
    df_plan['spot_energy_cost'] = spot_energy_cost
    df_plan['margin_cost'] = margin_cost
    df_plan['monthly_cost'] = cost_per_interval
    df_plan['total_cost'] = spot_energy_cost + margin_cost + cost_per_interval

    results.append(df_plan)

In [ ]:
# For loop to iterate through fixed plans and assign price to 15min consumption based on timestamp (day, night, weekend, national holiday)

for _, plan in df_fixed_plan.iterrows():
    effective_rate = np.where(
        df['is_night_rate'],
        plan['kwh_price_night'],
        plan['kwh_price_day']
    )
    
    # Interval energy cost
    interval_energy_cost = effective_rate * df['consumption_kwh']
    
    # Prorated monthly cost per 15min interval
    cost_per_interval = plan['monthly_price'] / (30.4375 * 24 * 4)
    
    # Total cost per interval = energy cost + standing charge portion
    interval_total_cost = interval_energy_cost + cost_per_interval
    
    # Build result dataframe for this plan
    df_plan = df[['timestamp', 'household_id', 'consumption_kwh']].copy()
    df_plan['plan_id'] = plan['plan_id']
    df_plan['plan_name'] = plan['electricity_plan_name']
    df_plan['vendor'] = plan['electricity_vendor_name']
    df_plan['plan_type'] = 'fixed'
    df_plan['total_cost'] = interval_total_cost
    df_plan['production_type'] = plan['production_type']
    #df_plan['contract_length_months'] = plan['contract_length_months']
    #df_plan['is_ending_fee'] = plan['is_ending_fee']
    
    results.append(df_plan)


In [ ]:
for _, plan in df_spot_plan.iterrows():
    # Effective rate
    effective_rate = df['spot_price_incl_vat'] + plan['margin_price']
    
    # Interval energy cost
    interval_energy_cost = effective_rate * df['consumption_kwh']
    
    # Prorated monthly cost per 15min interval
    cost_per_interval = plan['monthly_price'] / (30.4375 * 24 * 4)
    
    # Total cost per interval
    interval_total_cost = interval_energy_cost + cost_per_interval
    
    # Build result dataframe for this plan
    df_plan = df[['timestamp', 'household_id', 'consumption_kwh']].copy()
    df_plan['plan_id'] = plan['plan_id']
    df_plan['plan_name'] = plan['electricity_plan_name']
    df_plan['vendor'] = plan['electricity_vendor_name']
    df_plan['plan_type'] = 'spot'
    df_plan['total_cost'] = interval_total_cost
    df_plan['production_type'] = plan['production_type']
    #df_plan['is_ending_fee'] = plan['is_ending_fee']
    
    results.append(df_plan)


In [22]:
df_results = pd.concat(results, ignore_index=True)

In [24]:
df_results.sample(10)

,timestamp,household_id,consumption_kwh,plan_id,plan_name,vendor,plan_type,total_cost,production_type
86033405,2025-06-23 07:15:00,s-2,0.000,spot-9,Börsi Klõps,Elektrum Eesti OÜ,spot,0.000654,kombineeritud
179883579,2025-11-02 06:45:00,s-10,0.066,spot-4,Kodupakett + roheline,Alexela AS,spot,0.002611,roheline
51740440,2025-10-23 22:00:00,xl-7,0.942,fix-41,"Roheline Klõps (6, ööpäev)",Elektrum Eesti OÜ,fixed,0.120043,roheline
43204234,2025-09-06 02:30:00,s-7,0.000,fix-35,Kasulik Klõps (6),Elektrum Eesti OÜ,fixed,0.000654,kombineeritud
17231648,2025-06-24 08:00:00,m-6,0.004,fix-14,"Kindel (24, ööpäev)",AS Elenger Grupp,fixed,0.000509,tavaline
44328445,2025-05-02 15:15:00,s-3,0.000,fix-36,Roheline klõps (36),Elektrum Eesti OÜ,fixed,0.000654,roheline
194670503,2025-09-25 17:45:00,xl-8,0.029,spot-15,Marginaalivaba börsipakett S,Sunly Retail OÜ,spot,0.004499,roheline
58817697,2025-11-20 08:15:00,m-1,0.300,fix-47,Kindel 6,Enefit AS,fixed,0.038682,tavaline
14335335,2025-06-21 09:45:00,l-4,0.013,fix-12,"Kindel (36, ööpäev)",AS Elenger Grupp,fixed,0.001599,tavaline
134820211,2025-10-28 04:45:00,m-5,0.018,fix-28,Fiksteeritud hind + ühisarve (36),Electric Terminal OÜ,fixed,0.002555,tavaline


In [186]:
df_results.to_parquet("optimal_plan.parquet")

In [187]:
df_results.to_parquet(
    "optimal_plan_partitioned",
    partition_cols=["household_id"],
    index=False
)

In [184]:
#Data visualization

household = 's-3'
start_date = '2025-08-01'
end_date = '2026-01-01'

mask = (
    (df_results['household_id'] == household) &
    (df_results['timestamp'] >= start_date) &
    (df_results['timestamp'] < end_date)
)
df_filtered = df_results[mask]

df_filtered = df_filtered.copy()
df_filtered['month'] = df_filtered['timestamp'].dt.to_period('M').astype(str)

df_pivot = df_filtered.groupby(['plan_id', 'plan_name', 'vendor', 'plan_type', 'month'])['total_cost'].sum().reset_index()

df_pivot = df_pivot.pivot_table(
    index=['plan_id', 'plan_name', 'vendor', 'plan_type'],
    columns='month',
    values='total_cost'
).reset_index()


df_pivot.columns.name = None


month_columns = [col for col in df_pivot.columns if col not in ['plan_id', 'plan_name', 'vendor', 'plan_type']]

df_pivot['total'] = df_pivot[month_columns].sum(axis=1)

df_pivot = df_pivot.sort_values('total').reset_index(drop=True)

df_pivot[month_columns + ['total']] = df_pivot[month_columns + ['total']].round(2)


In [185]:
df_pivot

,plan_id,plan_name,vendor,plan_type,2025-08,2025-09,2025-10,2025-11,2025-12,total
0,spot-5,Muutuv (Elenger),AS Elenger Grupp,spot,0.82,0.93,1.02,1.02,0.92,4.72
1,fix-11,Kindel (36),AS Elenger Grupp,fixed,0.82,0.93,1.08,1.10,1.19,5.13
2,spot-6,Muutuv + roheline (Elenger),AS Elenger Grupp,spot,0.89,1.01,1.11,1.11,1.01,5.15
3,fix-13,Kindel (24),AS Elenger Grupp,fixed,0.85,0.97,1.12,1.14,1.23,5.30
4,fix-16,Kindel (12),AS Elenger Grupp,fixed,0.89,1.01,1.17,1.19,1.29,5.55
...,...,...,...,...,...,...,...,...,...,...
74,spot-14,Hinnalaega börsipakett + roheline,Enefit AS,spot,5.85,5.79,6.04,5.88,5.93,29.49
75,spot-3,Kodupakett,Alexela AS,spot,6.87,6.78,7.06,6.86,6.95,34.52
76,spot-17,Marginaalivaba börsipakett L,Sunly Retail OÜ,spot,6.88,6.79,7.07,6.87,6.96,34.57
77,spot-4,Kodupakett + roheline,Alexela AS,spot,6.91,6.82,7.11,6.91,7.00,34.75
